# Results for model: abacusai/dracarys-llama-3.1-70b-instruct

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
df = df.with_columns([
    pl.col('date_id').alias('batch'),
    pl.col('feature_00').alias('feature_00_raw')
])

df = df.with_columns([
    pl.col('feature_00_raw').over('batch').rank(method='dense', reverse=True).alias('feature_00_rank')
])

df = df.with_columns([
    pl.when(pl.col('feature_00_rank') > 0.85).then(pl.col('feature_00_raw')).otherwise(0).alias('TOP_QUANTILE')
])

# Prepare data for training
X = df.drop(['responder_6', 'date_id', 'feature_00_raw', 'feature_00_rank']).to_pandas()
y = df['responder_6'].to_pandas()

# Train XGBoost Regressor
model = xgb.XGBRegressor()
model.fit(X, y)